# BQR-DN V2 training on VOC2007

Trains the `bqr_dn_v2` method: official DINO R50 with region-aware pre-conditioning of training-only denoising queries. Normal detection and inference paths remain official DINO.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from gt_guided_dino.api import ExperimentConfig, train
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
config = ExperimentConfig(
    data_root=PROJECT_ROOT / 'VOC2007',
    output_root=PROJECT_ROOT / 'artifacts',
    method='bqr_dn_v2',
    epochs=12,
    train_limit=1000,
    lr_drop_epoch=11,
    bqr_enabled=True,
    bqr_dn_weight=1.0,
    bqr_points_per_level=4,
    bqr_gate_bias=-2.0,
)
config

In [ ]:
# Resumes artifacts/bqr_dn_v2/seed_42/checkpoints/latest.pt when present.
history = train(config, resume=True)
history[-1]

In [ ]:
import matplotlib.pyplot as plt
from gt_guided_dino.visualization import plot_history

plot_history(config.history_path);

epochs = [row['epoch'] for row in history]
figure, axes = plt.subplots(1, 3, figsize=(17, 4))
for axis, key, title in (
    (axes[0], 'bqr_gate_mean', 'BQR gate mean'),
    (axes[1], 'bqr_offset_abs_mean', 'Sampling offset magnitude'),
    (axes[2], 'bqr_attention_entropy', 'Sampling attention entropy'),
):
    axis.plot(epochs, [row.get(key, float('nan')) for row in history], marker='o')
    axis.set(title=title, xlabel='Epoch')
    axis.grid(alpha=0.25)
figure.tight_layout()